# Notebook 07 — Ensemble Strategy & Sentiment

**Phase 3 · Strategy Integration (1 / 2)**

---

## 🎯 Learning Objectives

| # | Goal |
|---|------|
| 1 | Understand **regime-weighted blending** of multiple sub-strategies |
| 2 | Build sub-strategy weight maps (momentum, mean-reversion, sector rotation) |
| 3 | Combine them using `_REGIME_WEIGHTS` blend factors |
| 4 | Apply a **sentiment overlay** (Fear & Greed Index) as a post-hoc multiplier |
| 5 | Implement **sector rotation** based on BTC dominance |
| 6 | Compare hand-written ensemble with the production `ensemble_combine()` |

### Prerequisites
- NB04 (Momentum), NB05 (Mean Reversion & Pairs), NB06 (Regime Detection)

In [ ]:
# ── Setup ──────────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True})
print("✅ Imports OK  |  Project root:", ROOT)

---
## 1 · The Ensemble Concept

No single strategy wins in all conditions. Our bot runs **four** sub-strategies simultaneously and combines their outputs with weights that depend on the **current regime** (from NB06):

```
┌──────────┐  ┌──────────────┐  ┌─────────────────┐  ┌────────────────┐
│ Momentum │  │ Mean-Revert  │  │ Sector Rotation │  │   Sentiment    │
│  weights │  │   weights    │  │     weights     │  │  (multiplier)  │
└────┬─────┘  └──────┬───────┘  └────────┬────────┘  └───────┬────────┘
     │               │                   │                   │
     │  ×blend_factor│  ×blend_factor    │  ×blend_factor    │
     └───────────────┼───────────────────┘                   │
                     ▼                                       │
              ┌──────────────┐                               │
              │  Sum weights │ ◄──── ×sentiment_multiplier ──┘
              └──────┬───────┘
                     ▼
              target_weights{symbol: weight}
```

### Regime Weights (from `bot/strategy/ensemble.py`)

| Regime | Momentum | Mean-Reversion | Sector Rotation | Sentiment | Cash |
|--------|----------|----------------|-----------------|-----------|------|
| **Bull** | 0.50 | 0.10 | 0.20 | 0.20 | 0% |
| **Ranging** | 0.20 | 0.50 | — | 0.30 | 0% |
| **Bear** | — | 0.30 | — | 0.20 | **50%** |

In [ ]:
# Regime weight table — exact copy from production
_REGIME_WEIGHTS = {
    "bull": {
        "momentum": 0.50,
        "sector_rotation": 0.20,
        "sentiment": 0.20,
        "mean_reversion": 0.10,
    },
    "ranging": {
        "mean_reversion": 0.50,
        "sentiment": 0.30,
        "momentum": 0.20,
    },
    "bear": {
        "mean_reversion": 0.30,
        "sentiment": 0.20,
        # remaining 50% is held as cash
    },
}

# Visualize
all_strategies = ["momentum", "mean_reversion", "sector_rotation", "sentiment", "cash"]
colours = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#95a5a6"]

data = []
for regime in ["bull", "ranging", "bear"]:
    w = _REGIME_WEIGHTS[regime]
    allocated = sum(w.values())
    row = [w.get(s, 0.0) for s in all_strategies[:-1]] + [max(0, 1.0 - allocated)]
    data.append(row)

weight_df = pd.DataFrame(data, index=["Bull", "Ranging", "Bear"], columns=all_strategies)
weight_df.plot.bar(figsize=(10, 5), color=colours, width=0.7, rot=0)
plt.title("Regime-Dependent Sub-Strategy Weights")
plt.ylabel("Weight")
plt.xlabel("Regime")
plt.legend(title="Strategy", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

weight_df

---
## 2 · Sub-Strategy Weight Maps

Each sub-strategy produces a **weight map** — a `dict[str, float]` mapping symbols to raw weights. Let's create mock outputs for a 6-asset universe.

In [ ]:
UNIVERSE = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT", "AVAXUSDT"]

# Momentum signal: top-3 by momentum score get weights
momentum_weights = {
    "SOLUSDT": 0.40,   # strongest momentum
    "ETHUSDT": 0.35,
    "AVAXUSDT": 0.25,
}

# Mean-reversion signal: oversold assets get weights
mean_reversion_weights = {
    "BTCUSDT": 0.50,   # most oversold
    "XRPUSDT": 0.30,
    "BNBUSDT": 0.20,
}

# Sector rotation: BTC dominance rising → heavy BTC/ETH
sector_rotation_weights = {
    "BTCUSDT": 0.40,
    "ETHUSDT": 0.25,
    "SOLUSDT": 0.125,
    "BNBUSDT": 0.125,
    "XRPUSDT": 0.05,
    "AVAXUSDT": 0.05,
}

print("Momentum:       ", momentum_weights)
print("Mean-Reversion: ", mean_reversion_weights)
print("Sector Rotation:", sector_rotation_weights)

---
## 3 · Step-by-Step Ensemble Blending

For a given regime, the ensemble for each symbol is:

$$
w_i^{\text{combined}} = \sum_{s \in \text{strategies}} \underbrace{w_i^{(s)}}_{\text{raw weight}} \times \underbrace{\alpha_s^{(\text{regime})}}_{\text{blend factor}}
$$

Then apply the sentiment multiplier $m$:

$$
w_i^{\text{final}} = w_i^{\text{combined}} \times \text{clamp}(m, 0.5, 1.5)
$$

In [ ]:
def ensemble_combine_manual(
    regime: str,
    momentum_w: dict[str, float],
    mrv_w: dict[str, float],
    sector_w: dict[str, float],
    sentiment_multiplier: float = 1.0,
) -> dict:
    """Hand-built ensemble to understand each step."""
    regime_key = regime.lower() if regime.lower() in _REGIME_WEIGHTS else "ranging"
    strat_weights = _REGIME_WEIGHTS[regime_key]
    
    # Step 1: Scale each signal map by its blend factor
    signal_maps = {
        "momentum": momentum_w,
        "mean_reversion": mrv_w,
        "sector_rotation": sector_w,
    }
    
    combined: dict[str, float] = {}
    contributions: dict[str, dict[str, float]] = {}
    
    for sig_name, w_map in signal_maps.items():
        blend = strat_weights.get(sig_name, 0.0)
        if blend <= 0:
            continue
        sig_contrib = {}
        for sym, raw_w in w_map.items():
            c = raw_w * blend
            sig_contrib[sym] = c
            combined[sym] = combined.get(sym, 0.0) + c
        contributions[sig_name] = sig_contrib
    
    # Step 2: Sentiment overlay
    sent_clamped = max(0.5, min(1.5, sentiment_multiplier))
    if sent_clamped != 1.0:
        combined = {s: w * sent_clamped for s, w in combined.items()}
    
    # Cash allocation
    allocated = sum(strat_weights.values())
    cash = max(0.0, 1.0 - allocated)
    
    return {
        "regime": regime_key,
        "target_weights": combined,
        "contributions": contributions,
        "cash_allocation": cash,
    }

# Run for all three regimes
for regime in ["bull", "ranging", "bear"]:
    result = ensemble_combine_manual(
        regime, momentum_weights, mean_reversion_weights, sector_rotation_weights
    )
    print(f"\n{'='*50}")
    print(f"Regime: {regime.upper()}  |  Cash: {result['cash_allocation']:.0%}")
    print(f"{'='*50}")
    tw = result['target_weights']
    for sym in sorted(tw, key=tw.get, reverse=True):
        print(f"  {sym:12s}: {tw[sym]:.4f}")
    print(f"  {'Total':12s}: {sum(tw.values()):.4f}")

---
## 4 · Visualizing Contribution Breakdown

A stacked bar chart shows which strategy contributes what to each symbol.

In [ ]:
regime = "bull"  # Change to see other regimes
result = ensemble_combine_manual(
    regime, momentum_weights, mean_reversion_weights, sector_rotation_weights
)

contribs = result["contributions"]
all_symbols = sorted(result["target_weights"].keys())

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(all_symbols))
strat_colours = {"momentum": "#3498db", "mean_reversion": "#e74c3c", "sector_rotation": "#2ecc71"}

for strat_name, colour in strat_colours.items():
    if strat_name not in contribs:
        continue
    vals = [contribs[strat_name].get(s, 0.0) for s in all_symbols]
    ax.bar(all_symbols, vals, bottom=bottom, label=strat_name, color=colour, alpha=0.8)
    bottom += vals

ax.set_title(f"Weight Contribution Breakdown — {regime.upper()} Regime")
ax.set_ylabel("Combined Weight")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5 · Sentiment Overlay: Fear & Greed Index

The [Alternative.me Fear & Greed Index](https://alternative.me/crypto/fear-and-greed-index/) provides a daily 0–100 sentiment reading:

| F&G Score | Label | Multiplier |
|-----------|-------|------------|
| 0–25 | Extreme Fear | 1.30 (contrarian buy) |
| 25–45 | Fear | 1.15 |
| 45–55 | Neutral | 1.00 |
| 55–75 | Greed | 0.85 |
| 75–100 | Extreme Greed | 0.70 (reduce exposure) |

The multiplier scales **all** combined weights proportionally.

In [ ]:
def fg_to_multiplier(fg_score: float) -> float:
    """Convert Fear & Greed score (0-100) to a sentiment multiplier."""
    if fg_score <= 25:
        return 1.30   # Extreme fear → contrarian bullish
    elif fg_score <= 45:
        return 1.15
    elif fg_score <= 55:
        return 1.00
    elif fg_score <= 75:
        return 0.85
    else:
        return 0.70   # Extreme greed → reduce exposure

# Sweep across F&G values
fg_values = range(0, 101, 5)
multipliers = [fg_to_multiplier(fg) for fg in fg_values]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fg_values, multipliers, marker="o", color="#e67e22", lw=2)
ax.axhline(1.0, color="gray", ls="--", lw=0.8)
ax.fill_between(fg_values, multipliers, 1.0,
                where=[m > 1.0 for m in multipliers], alpha=0.15, color="green", label="Boost")
ax.fill_between(fg_values, multipliers, 1.0,
                where=[m < 1.0 for m in multipliers], alpha=0.15, color="red", label="Reduce")
ax.set_xlabel("Fear & Greed Score")
ax.set_ylabel("Sentiment Multiplier")
ax.set_title("Fear & Greed → Sentiment Multiplier Mapping")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Effect of sentiment on portfolio in BULL regime
scenarios = [
    ("Extreme Fear (F&G=15)", 1.30),
    ("Neutral (F&G=50)", 1.00),
    ("Extreme Greed (F&G=85)", 0.70),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (label, mult) in zip(axes, scenarios):
    r = ensemble_combine_manual(
        "bull", momentum_weights, mean_reversion_weights,
        sector_rotation_weights, sentiment_multiplier=mult,
    )
    tw = r["target_weights"]
    syms = sorted(tw.keys())
    vals = [tw[s] for s in syms]
    bars = ax.bar(syms, vals, color="#3498db", alpha=0.8)
    ax.set_title(f"{label}\n(mult={mult:.2f})")
    ax.set_ylim(0, 0.35)
    ax.tick_params(axis="x", rotation=45)
    total = sum(vals)
    ax.axhline(total / len(syms), ls="--", color="red", lw=0.8)

axes[0].set_ylabel("Weight")
fig.suptitle("Sentiment Impact on BULL Regime Portfolio", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6 · Sector Rotation via BTC Dominance

BTC dominance (% of total crypto market cap held by Bitcoin) signals capital flow:

| Dominance Trend | Rotation Regime | Strategy |
|-----------------|-----------------|----------|
| Rising (Δ ≥ 0.5%) | `bitcoin_led` | Heavy BTC/ETH |
| Falling (Δ ≤ -0.5%) | `altcoin_rotation` | Heavy alts |
| Flat | `neutral` | Balanced |

In [ ]:
# Symbol classification (from production)
_BTC_SYMBOLS = frozenset({"BTCUSDT", "BTCUSD"})
_ETH_SYMBOLS = frozenset({"ETHUSDT", "ETHUSD"})
_LARGE_ALT_PREFIXES = ("SOL", "BNB", "XRP", "ADA", "AVAX", "DOT", "MATIC", "LINK", "DOGE")

def classify_symbol(symbol: str) -> str:
    upper = symbol.upper()
    if upper in _BTC_SYMBOLS: return "btc"
    if upper in _ETH_SYMBOLS: return "eth"
    for prefix in _LARGE_ALT_PREFIXES:
        if upper.startswith(prefix): return "large_alt"
    return "small_alt"

# Sector allocation tables
SECTOR_ALLOCATIONS = {
    "bitcoin_led":      {"btc": 0.40, "eth": 0.25, "large_alt": 0.25, "small_alt": 0.10},
    "altcoin_rotation": {"btc": 0.15, "eth": 0.20, "large_alt": 0.40, "small_alt": 0.25},
    "neutral":          {"btc": 0.30, "eth": 0.25, "large_alt": 0.30, "small_alt": 0.15},
}

# Visualize allocations
alloc_df = pd.DataFrame(SECTOR_ALLOCATIONS).T
alloc_df.plot.bar(figsize=(10, 5), width=0.7,
                  color=["#f1c40f", "#8e44ad", "#3498db", "#e74c3c"], rot=0)
plt.title("Sector Allocation by BTC Dominance Regime")
plt.ylabel("Allocation Weight")
plt.legend(title="Sector")
plt.tight_layout()
plt.show()

In [ ]:
def sector_rotation_weights_manual(
    universe: list[str],
    btc_dominance: float,
    prev_dominance: float,
    min_change: float = 0.5,
) -> dict[str, float]:
    """Compute per-asset weights from BTC dominance changes."""
    delta = btc_dominance - prev_dominance
    if delta >= min_change:
        regime = "bitcoin_led"
    elif delta <= -min_change:
        regime = "altcoin_rotation"
    else:
        regime = "neutral"
    
    allocs = SECTOR_ALLOCATIONS[regime]
    
    # Group symbols into buckets
    buckets: dict[str, list[str]] = {"btc": [], "eth": [], "large_alt": [], "small_alt": []}
    for sym in universe:
        buckets[classify_symbol(sym)].append(sym)
    
    # Equal weight within each bucket
    weights = {}
    for bucket, syms in buckets.items():
        if not syms:
            continue
        per_asset = allocs[bucket] / len(syms)
        for sym in syms:
            weights[sym] = per_asset
    
    return weights

# Example: BTC dominance rising
universe = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT", "AVAXUSDT", "DOGEUSDT"]
for dom, prev, label in [(55.0, 53.0, "BTC-led (+2%)"), (50.0, 50.3, "Neutral"), (48.0, 50.0, "Alt rotation (-2%)")]:
    w = sector_rotation_weights_manual(universe, dom, prev)
    print(f"\n{label}:")
    for sym in sorted(w, key=w.get, reverse=True):
        print(f"  {sym:12s}: {w[sym]:.3f}")

---
## 7 · Full Ensemble Example Across Regimes

Let's trace the complete pipeline: regime → blend → sentiment → final weights.

In [ ]:
# Simulate a 30-day period with changing regimes and sentiment
np.random.seed(42)
days = 30
regimes = ["bull"] * 12 + ["ranging"] * 10 + ["bear"] * 8
fg_scores = np.clip(50 + np.cumsum(np.random.randn(days) * 5), 5, 95)

# Track total equity allocation per day (sum of non-cash weights)
daily_allocation = []
daily_cash = []
daily_top_asset = []

for i in range(days):
    mult = fg_to_multiplier(fg_scores[i])
    r = ensemble_combine_manual(
        regimes[i], momentum_weights, mean_reversion_weights,
        sector_rotation_weights, sentiment_multiplier=mult,
    )
    tw = r["target_weights"]
    daily_allocation.append(sum(tw.values()))
    daily_cash.append(r["cash_allocation"])
    daily_top_asset.append(max(tw, key=tw.get) if tw else "N/A")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Regime
regime_colours = {"bull": "#2ecc71", "ranging": "#f39c12", "bear": "#e74c3c"}
for i in range(days):
    ax1.axvspan(i - 0.5, i + 0.5, alpha=0.3, color=regime_colours[regimes[i]])
ax1.set_ylabel("Regime")
ax1.set_yticks([])
patches = [mpatches.Patch(color=c, alpha=0.3, label=r) for r, c in regime_colours.items()]
ax1.legend(handles=patches, loc="upper right")
ax1.set_title("30-Day Ensemble Simulation")

# F&G score
ax2.plot(range(days), fg_scores, color="#e67e22", lw=2)
ax2.axhline(50, color="gray", ls="--", lw=0.8)
ax2.fill_between(range(days), fg_scores, 50,
                 where=fg_scores < 50, alpha=0.15, color="green")
ax2.fill_between(range(days), fg_scores, 50,
                 where=fg_scores > 50, alpha=0.15, color="red")
ax2.set_ylabel("F&G Score")
ax2.set_ylim(0, 100)

# Equity allocation
ax3.bar(range(days), daily_allocation, color="#3498db", alpha=0.7, label="Equity")
ax3.bar(range(days), daily_cash, bottom=daily_allocation, color="#95a5a6", alpha=0.5, label="Cash")
ax3.set_ylabel("Total Allocation")
ax3.set_xlabel("Day")
ax3.legend()

plt.tight_layout()
plt.show()

---
## 8 · Production Code: `ensemble_combine()`

In [ ]:
from bot.strategy.ensemble import ensemble_combine, _REGIME_WEIGHTS as prod_weights

# Compare with our manual implementation
for regime in ["bull", "ranging", "bear"]:
    prod_result = ensemble_combine(
        regime,
        momentum_weights=momentum_weights,
        mean_reversion_weights=mean_reversion_weights,
        sector_rotation_weights=sector_rotation_weights,
        sentiment_multiplier=1.0,
    )
    manual_result = ensemble_combine_manual(
        regime, momentum_weights, mean_reversion_weights,
        sector_rotation_weights, sentiment_multiplier=1.0,
    )
    
    match = all(
        abs(prod_result.target_weights.get(s, 0) - manual_result["target_weights"].get(s, 0)) < 1e-9
        for s in set(list(prod_result.target_weights) + list(manual_result["target_weights"]))
    )
    print(f"{regime:8s} — match: {'✅' if match else '❌'}  |  cash: {prod_result.cash_allocation:.0%}")

In [ ]:
# Production sector rotation
from bot.signals.sector_rotation import sector_rotation_weights as prod_sector_weights

prod_sw = prod_sector_weights(universe, btc_dominance=55.0, previous_dominance=53.0)
manual_sw = sector_rotation_weights_manual(universe, 55.0, 53.0)

print("Production vs Manual sector weights:")
for sym in sorted(prod_sw.keys()):
    p = prod_sw.get(sym, 0)
    m = manual_sw.get(sym, 0)
    status = "✅" if abs(p - m) < 1e-9 else "❌"
    print(f"  {sym:12s}: prod={p:.4f}  manual={m:.4f}  {status}")

---
## 9 · Sensitivity: How Regime Weights Affect Returns

What if we change the regime weight allocations? Let's simulate.

In [ ]:
# Simulate daily returns for each strategy type
np.random.seed(123)
n_days = 200

# Bull: momentum does well; Bear: mean-reversion survives
strategy_returns = {
    "momentum":       np.random.normal(0.003, 0.02, n_days),
    "mean_reversion": np.random.normal(0.001, 0.01, n_days),
    "sector_rotation":np.random.normal(0.002, 0.015, n_days),
}

# Make momentum bad in bear market (last 50 days)
strategy_returns["momentum"][-50:] = np.random.normal(-0.005, 0.03, 50)
strategy_returns["mean_reversion"][-50:] = np.random.normal(0.002, 0.01, 50)

# Regime sequence
sim_regimes = ["bull"] * 80 + ["ranging"] * 70 + ["bear"] * 50

# 1. Static equal weight (no regime adaptation)
static_returns = []
for i in range(n_days):
    r = sum(strategy_returns[s][i] * (1/3) for s in strategy_returns)
    static_returns.append(r)

# 2. Regime-adaptive weights
adaptive_returns = []
for i in range(n_days):
    rw = _REGIME_WEIGHTS[sim_regimes[i]]
    r = sum(strategy_returns[s][i] * rw.get(s, 0.0) for s in strategy_returns)
    adaptive_returns.append(r)

static_equity  = np.exp(np.cumsum(static_returns))
adaptive_equity = np.exp(np.cumsum(adaptive_returns))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(static_equity, label="Static Equal Weight", lw=1.5, alpha=0.8)
ax.plot(adaptive_equity, label="Regime-Adaptive", lw=1.5, alpha=0.8)

# Shade regime periods
for i in range(n_days):
    ax.axvspan(i - 0.5, i + 0.5, alpha=0.05, color=regime_colours[sim_regimes[i]])

ax.set_title("Static vs Regime-Adaptive Ensemble")
ax.set_xlabel("Day")
ax.set_ylabel("Equity (log scale)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Static Final Equity:   {static_equity[-1]:.4f}")
print(f"Adaptive Final Equity: {adaptive_equity[-1]:.4f}")
print(f"Advantage:             {(adaptive_equity[-1]/static_equity[-1] - 1)*100:.1f}%")

---
## 10 · Key Takeaways

| Concept | Detail |
|---------|--------|
| **Ensemble Blending** | Scale each sub-strategy's weights by regime-dependent blend factor, then sum |
| **Regime Weights** | Bull favours momentum (0.50); Ranging favours mean-reversion (0.50); Bear holds 50% cash |
| **Sentiment Overlay** | Fear & Greed Index → multiplier [0.5, 1.5] applied post-blend |
| **Sector Rotation** | BTC dominance changes → heavy BTC/ETH vs heavy alts |
| **Cash Allocation** | Bear regime explicitly withholds 50% as cash — no signal can override this |

### Ensemble Pipeline

```
detect_regime() → regime (bull / ranging / bear)
    │
    ├── _REGIME_WEIGHTS[regime] → blend_factors
    │
    ├── momentum_weights × blend_factor["momentum"]
    ├── mrv_weights × blend_factor["mean_reversion"]
    ├── sector_weights × blend_factor["sector_rotation"]
    │
    ├── SUM → combined_weights
    │
    ├── × sentiment_multiplier(F&G)
    │
    └── target_weights + cash_allocation
```

---
## 🔬 Exercises

1. **Alternative Regime Weights:** Design your own `_REGIME_WEIGHTS` with a 4th regime "volatile_bull" where momentum=0.30, mrv=0.20, sector=0.20, sentiment=0.30. Test it.

2. **Dynamic Sentiment:** Instead of discrete buckets, use a linear mapping: `multiplier = 1.5 - (fg_score / 100)`. How does this change portfolio behaviour?

3. **Weighted Average vs Max:** Replace summation with a max-vote approach where each symbol keeps only its highest contributor. Compare results.

4. **Backtest the Ensemble:** Using NB09's backtester, compare regime-adaptive vs static-weight ensemble on 1 year of BTC data.

---
## ✅ Knowledge Check

1. Why does the bear regime hold 50% in cash rather than allocating it to mean-reversion?
2. How does the sentiment multiplier interact with cash allocation? (Hint: it doesn't.)
3. If momentum_weights has 3 assets and sector_rotation_weights has 6, how many assets appear in the combined output?
4. What happens if you pass an unknown regime (e.g., "crash") to `ensemble_combine()`?
5. Why is the sentiment multiplier clamped to [0.5, 1.5]?

---
## 🔗 Next

**[NB08 — Portfolio Optimization & Risk Management →](08_Portfolio_Optimization_and_Risk.ipynb)**

We'll take the `target_weights` from the ensemble and pass them through weight normalization, position limits, cash floors, and circuit breakers.